# 04 — Backtest Analysis

**OandaFX** — Walk-forward backtest with full performance analysis.

**Loads trained model artifacts and processed data from Google Drive** (saved by notebooks 02 & 03).

**Run on Google Colab or locally.**

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SETUP
# ═══════════════════════════════════════════════════════════════════
import sys, os, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = Path('/content/drive/MyDrive/oandafx')

    # Clone the repository
    if not Path('/content/wavebot-v1').exists():
        !git clone https://github.com/Unseengap/wavebot-v1.git /content/wavebot-v1
    sys.path.insert(0, '/content/wavebot-v1')

    # Install dependencies
    !pip install -q torch stable-baselines3 gymnasium pandas-ta scikit-learn matplotlib plotly tqdm
else:
    DRIVE_BASE = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
    sys.path.insert(0, str(DRIVE_BASE))

# All data is loaded from Drive (saved by notebooks 01, 02, 03)
PROCESSED_DIR = DRIVE_BASE / 'data' / 'processed'
MODELS_DIR = DRIVE_BASE / 'models'
CHECKPOINT_DIR = DRIVE_BASE / 'checkpoints'
REPORTS_DIR = DRIVE_BASE / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify required files exist on Drive
required = [
    (PROCESSED_DIR / 'all_pairs_features.parquet', 'Run notebook 02'),
    (CHECKPOINT_DIR / 'split_indices.json', 'Run notebook 03'),
]
all_ok = True
for path, fix in required:
    if path.exists():
        print(f'✅ {path.name}')
    else:
        print(f'❌ Missing: {path.name} — {fix} first')
        all_ok = False

if not all_ok:
    print('\n⚠️  Some required files are missing. Run prior notebooks first.')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LOAD MODEL & DATA
# ═══════════════════════════════════════════════════════════════════
from src.models.transformer import ForexTransformer
from stable_baselines3 import PPO
import pickle

# Select model version (use 'latest' or a specific version)
MODEL_VERSION = 'latest'  # or 'v1.0.0'

model_dir = MODELS_DIR / MODEL_VERSION
if model_dir.is_symlink():
    model_dir = model_dir.resolve()

print(f'Loading model from: {model_dir}')

# Load metadata
with open(model_dir / 'metadata.json', 'r') as f:
    metadata = json.load(f)
print(f'Model version: {metadata["version"]}')
print(f'Training date: {metadata["training_date"]}')
print(f'Pairs: {metadata["pairs_trained_on"]}')

# Load feature columns
with open(model_dir / 'feature_columns.json', 'r') as f:
    feature_cols = json.load(f)
print(f'Features: {len(feature_cols)}')

# Load Transformer
transformer = ForexTransformer(
    input_dim=len(feature_cols), d_model=256, nhead=8,
    num_layers=4, dim_feedforward=1024, dropout=0.1,
    num_classes=3, num_horizons=3,
).to(device)
transformer.load_state_dict(torch.load(model_dir / 'transformer.pt', map_location=device))
transformer.eval()
print(f'Transformer: {sum(p.numel() for p in transformer.parameters()):,} params')

# Load PPO agent
ppo_agent = PPO.load(str(model_dir / 'ppo_agent'), device=device)
print('PPO agent loaded')

# Load scaler
with open(model_dir / 'sl_tp_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Load data & split
all_features = pd.read_parquet(PROCESSED_DIR / 'all_pairs_features.parquet')
with open(CHECKPOINT_DIR / 'split_indices.json', 'r') as f:
    split_info = json.load(f)

val_end = split_info['val_end']
test_df = all_features.iloc[val_end:].copy()
print(f'\nTest set: {len(test_df):,} rows | {test_df["time"].min()} → {test_df["time"].max()}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  RUN WALK-FORWARD BACKTEST
# ═══════════════════════════════════════════════════════════════════
from src.backtest.engine import WalkForwardBacktest
from src.backtest.metrics import compute_metrics
from src.models.sl_tp_module import DynamicSLTP

sl_tp_module = DynamicSLTP()

# Run backtest per pair
pairs = test_df['pair'].unique() if 'pair' in test_df.columns else ['ALL']
all_trades = []
pair_metrics = {}

for pair in pairs:
    print(f'\nBacktesting {pair}...')
    pair_data = test_df[test_df['pair'] == pair] if 'pair' in test_df.columns else test_df
    
    bt = WalkForwardBacktest(
        features_df=pair_data,
        feature_cols=feature_cols,
        transformer=transformer,
        sl_tp_module=sl_tp_module,
        spread_pips=1.2,
        initial_balance=10_000,
    )
    
    trades = bt.run()
    for t in trades:
        t['pair'] = pair
    all_trades.extend(trades)
    
    m = compute_metrics(trades)
    pair_metrics[pair] = m
    print(f'  {pair}: {m["total_trades"]} trades, '
          f'Sharpe={m["sharpe"]:.2f}, WR={m["win_rate"]:.1%}, '
          f'PF={m["profit_factor"]:.2f}, MDD={m["max_drawdown"]:.1%}')

# Aggregate metrics
agg_metrics = compute_metrics(all_trades)
print(f'\n{"=" * 60}')
print(f'AGGREGATE: {agg_metrics["total_trades"]} trades, '
      f'Sharpe={agg_metrics["sharpe"]:.2f}, WR={agg_metrics["win_rate"]:.1%}, '
      f'PF={agg_metrics["profit_factor"]:.2f}, MDD={agg_metrics["max_drawdown"]:.1%}')

trades_df = pd.DataFrame(all_trades)
print(f'\nTrades DataFrame: {trades_df.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  EQUITY CURVE
# ═══════════════════════════════════════════════════════════════════
initial_balance = 10_000
trades_df = trades_df.sort_values('exit_time').reset_index(drop=True)
trades_df['cumulative_pnl'] = trades_df['pnl'].cumsum()
trades_df['equity'] = initial_balance + trades_df['cumulative_pnl']

# Compute drawdown
trades_df['peak'] = trades_df['equity'].cummax()
trades_df['drawdown'] = (trades_df['equity'] - trades_df['peak']) / trades_df['peak']

fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1]})

# Equity curve
ax1 = axes[0]
ax1.plot(trades_df['exit_time'], trades_df['equity'], color='#00ff88', linewidth=1.5, label='Equity')
ax1.axhline(y=initial_balance, color='white', linestyle='--', alpha=0.3, label='Starting Balance')
ax1.fill_between(trades_df['exit_time'], initial_balance, trades_df['equity'],
                 where=trades_df['equity'] >= initial_balance, alpha=0.15, color='#00ff88')
ax1.fill_between(trades_df['exit_time'], initial_balance, trades_df['equity'],
                 where=trades_df['equity'] < initial_balance, alpha=0.15, color='#ff4444')
ax1.set_title('Equity Curve (Walk-Forward Backtest)', fontsize=14)
ax1.set_ylabel('Equity ($)')
ax1.legend()
ax1.grid(True, alpha=0.2)

# Drawdown
ax2 = axes[1]
ax2.fill_between(trades_df['exit_time'], 0, trades_df['drawdown'] * 100,
                 color='#ff4444', alpha=0.4)
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {REPORTS_DIR / "equity_curve.png"}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  MONTHLY RETURNS HEATMAP
# ═══════════════════════════════════════════════════════════════════
trades_df['exit_month'] = pd.to_datetime(trades_df['exit_time']).dt.to_period('M')
monthly = trades_df.groupby('exit_month')['pnl'].sum()

# Create year × month matrix
monthly_idx = monthly.index.to_timestamp()
years = sorted(set(monthly_idx.year))
months = list(range(1, 13))
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

returns_matrix = np.full((len(years), 12), np.nan)
for dt, pnl_val in zip(monthly_idx, monthly.values):
    yi = years.index(dt.year)
    mi = dt.month - 1
    returns_matrix[yi, mi] = pnl_val

fig, ax = plt.subplots(figsize=(14, max(3, len(years) * 0.8)))
im = ax.imshow(returns_matrix, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(12))
ax.set_xticklabels(month_names)
ax.set_yticks(range(len(years)))
ax.set_yticklabels(years)

# Annotate cells
for i in range(len(years)):
    for j in range(12):
        val = returns_matrix[i, j]
        if not np.isnan(val):
            color = 'black' if abs(val) < np.nanmax(np.abs(returns_matrix)) * 0.5 else 'white'
            ax.text(j, i, f'${val:.0f}', ha='center', va='center', color=color, fontsize=9)

ax.set_title('Monthly P&L Heatmap', fontsize=14)
plt.colorbar(im, label='P&L ($)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'monthly_returns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  TRADE DISTRIBUTION ANALYSIS
# ═══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# P&L distribution
ax = axes[0, 0]
winners = trades_df[trades_df['pnl'] > 0]['pnl']
losers = trades_df[trades_df['pnl'] <= 0]['pnl']
ax.hist(winners, bins=50, color='#00ff88', alpha=0.7, label=f'Winners ({len(winners)})')
ax.hist(losers, bins=50, color='#ff4444', alpha=0.7, label=f'Losers ({len(losers)})')
ax.axvline(x=0, color='white', linestyle='--', alpha=0.5)
ax.set_xlabel('P&L ($)')
ax.set_title('P&L Distribution')
ax.legend()
ax.grid(True, alpha=0.2)

# MFE vs MAE (if available)
ax = axes[0, 1]
if 'mfe' in trades_df.columns and 'mae' in trades_df.columns:
    colors = ['#00ff88' if p > 0 else '#ff4444' for p in trades_df['pnl']]
    ax.scatter(trades_df['mae'], trades_df['mfe'], c=colors, alpha=0.4, s=10)
    ax.set_xlabel('MAE (Max Adverse Excursion)')
    ax.set_ylabel('MFE (Max Favorable Excursion)')
    ax.set_title('MFE vs MAE')
else:
    ax.text(0.5, 0.5, 'MFE/MAE not available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('MFE vs MAE (N/A)')
ax.grid(True, alpha=0.2)

# Trade duration histogram
ax = axes[1, 0]
if 'duration_bars' in trades_df.columns:
    ax.hist(trades_df['duration_bars'], bins=50, color='#4488ff', alpha=0.7)
    ax.axvline(x=trades_df['duration_bars'].median(), color='yellow', linestyle='--',
               label=f'Median: {trades_df["duration_bars"].median():.0f} bars')
    ax.set_xlabel('Duration (bars)')
    ax.set_title('Trade Duration Distribution')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'Duration not available', ha='center', va='center', transform=ax.transAxes)
ax.grid(True, alpha=0.2)

# Win rate by pair
ax = axes[1, 1]
if 'pair' in trades_df.columns:
    pair_wr = trades_df.groupby('pair').apply(
        lambda g: (g['pnl'] > 0).mean(), include_groups=False
    ).sort_values(ascending=True)
    colors_wr = ['#00ff88' if wr > 0.5 else '#ff4444' for wr in pair_wr.values]
    ax.barh(pair_wr.index, pair_wr.values * 100, color=colors_wr, alpha=0.8)
    ax.axvline(x=50, color='white', linestyle='--', alpha=0.3)
    ax.set_xlabel('Win Rate (%)')
    ax.set_title('Win Rate by Pair')
else:
    ax.text(0.5, 0.5, 'No pair data', ha='center', va='center', transform=ax.transAxes)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'trade_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SESSION & DAY-OF-WEEK ANALYSIS
# ═══════════════════════════════════════════════════════════════════
trades_df['entry_time'] = pd.to_datetime(trades_df['entry_time'])
trades_df['entry_hour'] = trades_df['entry_time'].dt.hour
trades_df['entry_dow'] = trades_df['entry_time'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# P&L by entry hour
ax = axes[0]
hourly_pnl = trades_df.groupby('entry_hour')['pnl'].sum()
colors_h = ['#00ff88' if v > 0 else '#ff4444' for v in hourly_pnl.values]
ax.bar(hourly_pnl.index, hourly_pnl.values, color=colors_h, alpha=0.8)
ax.set_xlabel('Entry Hour (UTC)')
ax.set_ylabel('Total P&L ($)')
ax.set_title('P&L by Entry Hour')

# Highlight trading sessions
ax.axvspan(7, 16, alpha=0.05, color='blue', label='London')
ax.axvspan(13, 21, alpha=0.05, color='green', label='New York')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# P&L by day of week
ax = axes[1]
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
daily_pnl = trades_df.groupby('entry_dow')['pnl'].sum().reindex(day_order)
colors_d = ['#00ff88' if v > 0 else '#ff4444' for v in daily_pnl.values]
ax.bar(range(len(day_order)), daily_pnl.values, color=colors_d, alpha=0.8)
ax.set_xticks(range(len(day_order)))
ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri'])
ax.set_ylabel('Total P&L ($)')
ax.set_title('P&L by Day of Week')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'session_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  COMPREHENSIVE METRICS TABLE
# ═══════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('COMPREHENSIVE BACKTEST REPORT')
print('=' * 70)
print(f'Model Version: {metadata["version"]}')
print(f'Test Period:   {test_df["time"].min()} → {test_df["time"].max()}')
print(f'Pairs:         {list(pairs)}')
print('-' * 70)

m = agg_metrics
rows = [
    ('Total Return',        f'{m["total_return"]:.2%}'),
    ('Annualized Return',   f'{m["annualized_return"]:.2%}'),
    ('Sharpe Ratio',        f'{m["sharpe"]:.2f}'),
    ('Sortino Ratio',       f'{m["sortino"]:.2f}'),
    ('Max Drawdown',        f'{m["max_drawdown"]:.2%}'),
    ('Win Rate',            f'{m["win_rate"]:.2%}'),
    ('Profit Factor',       f'{m["profit_factor"]:.2f}'),
    ('Total Trades',        f'{m["total_trades"]}'),
    ('Avg Win',             f'${m.get("avg_win", 0):.2f}'),
    ('Avg Loss',            f'${m.get("avg_loss", 0):.2f}'),
    ('Avg Duration',        f'{m.get("avg_duration_bars", 0):.0f} bars'),
    ('Max Consecutive Wins',  f'{m.get("max_consec_wins", 0)}'),
    ('Max Consecutive Losses', f'{m.get("max_consec_losses", 0)}'),
    ('Avg MFE',             f'{m.get("avg_mfe", 0):.2f}'),
    ('Avg MAE',             f'{m.get("avg_mae", 0):.2f}'),
]

for label, value in rows:
    print(f'  {label:<28} {value}')

# Deployment readiness
print('\n' + '-' * 70)
THRESHOLDS = {'sharpe': 1.0, 'max_drawdown': -0.15, 'profit_factor': 1.2, 'win_rate': 0.45}
print('DEPLOYMENT THRESHOLDS:')
checks = {
    'Sharpe ≥ 1.0': m['sharpe'] >= THRESHOLDS['sharpe'],
    'Max DD ≥ -15%': m['max_drawdown'] >= THRESHOLDS['max_drawdown'],
    'PF ≥ 1.2': m['profit_factor'] >= THRESHOLDS['profit_factor'],
    'WR ≥ 45%': m['win_rate'] >= THRESHOLDS['win_rate'],
}
for check, passed in checks.items():
    icon = '✅' if passed else '❌'
    print(f'  {icon} {check}')

all_pass = all(checks.values())
print(f'\n{"=" * 70}')
if all_pass:
    print('✅ MODEL APPROVED FOR DEPLOYMENT')
else:
    print('❌ MODEL NOT APPROVED — review failing thresholds above')
print(f'{"=" * 70}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PER-PAIR METRICS TABLE
# ═══════════════════════════════════════════════════════════════════
print('\nPER-PAIR BREAKDOWN:')
print(f'{"Pair":<12} {"Trades":>7} {"Win Rate":>9} {"Sharpe":>8} {"PF":>6} {"MaxDD":>8} {"P&L":>10}')
print('-' * 70)
for pair, pm in sorted(pair_metrics.items()):
    pair_pnl = trades_df[trades_df['pair'] == pair]['pnl'].sum()
    print(f'{pair:<12} {pm["total_trades"]:>7} {pm["win_rate"]:>8.1%} '
          f'{pm["sharpe"]:>8.2f} {pm["profit_factor"]:>6.2f} '
          f'{pm["max_drawdown"]:>7.1%} {pair_pnl:>10.2f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SAVE REPORT TO GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════════
from src.backtest.reporter import generate_html_report
from datetime import datetime

# Save trades CSV
csv_path = REPORTS_DIR / f'trades_{metadata["version"]}_{datetime.utcnow().strftime("%Y%m%d")}.csv'
trades_df.to_csv(csv_path, index=False)
print(f'Trades CSV saved: {csv_path}')

# Save metrics JSON
metrics_output = {
    'model_version': metadata['version'],
    'test_period': [str(test_df['time'].min()), str(test_df['time'].max())],
    'aggregate_metrics': agg_metrics,
    'per_pair_metrics': pair_metrics,
    'deployment_approved': all_pass,
    'generated_at': datetime.utcnow().isoformat(),
}
metrics_json_path = REPORTS_DIR / f'metrics_{metadata["version"]}_{datetime.utcnow().strftime("%Y%m%d")}.json'
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_output, f, indent=2, default=str)
print(f'Metrics JSON saved: {metrics_json_path}')

# Generate HTML report
try:
    html_path = REPORTS_DIR / f'report_{metadata["version"]}_{datetime.utcnow().strftime("%Y%m%d")}.html'
    generate_html_report(all_trades, agg_metrics, str(html_path))
    print(f'HTML report saved: {html_path}')
except Exception as e:
    print(f'HTML report generation skipped: {e}')

print(f'\n✅ All reports saved to: {REPORTS_DIR}')
print(f'Files: {[f.name for f in REPORTS_DIR.iterdir()]}')